# 05 - Retrieval Profiles (Hue Foods RAG MVP)

Notebook này chạy **runtime thật** của Phase 5: ba retrieval profiles `dense_only`, `hybrid_no_rerank` và `hybrid_rerank` trên cùng một câu hỏi Hue Foods, dùng candidate collection `hue_foods_e5_small_384_dense` trong Qdrant thật, E5 thật và MiniLM thật. Không sửa file config: mỗi profile được chọn bằng deep copy settings trong memory.

**Prerequisite**

- Qdrant local đang chạy (candidate collection `hue_foods_e5_small_384_dense`, 572 points).
- `intfloat/multilingual-e5-small` và `cross-encoder/ms-marco-MiniLM-L-6-v2` sẵn sàng trên local CPU.
- Không tốn phí: không gọi OpenAI, OpenRouter hoặc model API nào.

**Kết quả mong đợi khi Run All**

- Mỗi profile in status (profile, collection, points, bm25_ready, reranker_ready), số documents retrieved, top-3 sources và score fields hợp lệ của stage đã chạy, context character count.
- Không in full context, không benchmark, không claim winner.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError(
        "Khong tim thay thu muc backend/. Hay mo notebook nay tu repo root "
        "hoac tu thu muc notebooks/."
    )
print(f"backend on path: {sys.path[0]}")


## Cấu hình retrieval và reranking

Nhóm `retrieval` khai báo depth, fusion weights và context budget; nhóm `reranking` khai báo MiniLM local. `active_profile` trong file config vẫn là `dense_only` - notebook không đổi file này.


In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
retrieval = settings["retrieval"]
reranking = settings["reranking"]
print("active_profile (trong config):", settings["active_profile"])
print("retrieval.top_k:", retrieval["top_k"])
print("retrieval.candidate_multiplier:", retrieval["candidate_multiplier"])
print("dense/bm25 weights:", retrieval["dense_weight"], retrieval["bm25_weight"])
print("context limits:", retrieval["max_context_documents"], "docs /", retrieval["max_context_characters"], "chars")
print("reranking.model:", reranking["model"])
print("reranking.device:", reranking["device"])
print("reranking.top_k:", reranking["top_k"])


## Ba profiles và stage thực sự chạy

| Profile | Stage |
|---|---|
| `dense_only` | Qdrant dense top 10 (E5 query embedding) |
| `hybrid_no_rerank` | dense candidates 30 -> Python BM25 -> min-max fusion 0.6/0.4 -> top 10 |
| `hybrid_rerank` | hybrid pipeline -> local MiniLM rerank 10 pairs -> top 5 |

ContextBuilder ghép tối đa 5 whole chunks thành các khối dễ đọc gồm Tiêu đề, Mục và Nội dung. Builder trả thẳng một chuỗi; source path và score không đi vào public answer contract.


In [ ]:
import copy
import time

from retrieval.context_builder import ContextBuilder
from retrieval.service import build_service

question = "Bún bò Huế có gì đặc biệt?"
candidate_collection = "hue_foods_e5_small_384_dense"

builder = ContextBuilder(
    max_documents=settings["retrieval"]["max_context_documents"],
    max_characters=settings["retrieval"]["max_context_characters"],
)


def run_profile(profile_name, expected_fields):
    profile_settings = copy.deepcopy(settings)
    profile_settings["active_profile"] = profile_name
    profile_settings["vector_database"]["collection_name"] = candidate_collection
    service = build_service(profile_settings)
    started = time.monotonic()
    documents = service.search(question)
    elapsed = round(time.monotonic() - started, 1)
    status = service.status
    print("=== profile:", status.active_profile)
    print("collection:", status.collection_name, "| points:", status.point_count,
          "| bm25_ready:", status.bm25_ready, "| reranker_ready:", status.reranker_ready)
    print("retrieved documents:", len(documents), "| search seconds:", elapsed)
    for rank, doc in enumerate(documents[:3], start=1):
        print(f"  {rank}. {doc.metadata['chunk_id']} | score={round(doc.score, 4)}")
    top = documents[0]
    missing = expected_fields - set(top.metadata)
    assert not missing, f"missing score fields: {missing}"
    print("top-1 score fields:", sorted(
        k for k in top.metadata if k.endswith("_score") or k.startswith("retrieval")
    ))
    context = builder.build(documents)
    print("context characters:", len(context))
    return service


## Profile `dense_only`

Chỉ chạy dense retrieval: E5 query embedding thật -> Qdrant dense top 10. Không fit BM25, không load reranker.


In [ ]:
service_dense = run_profile("dense_only", expected_fields={"dense_score"})


## Profile `hybrid_no_rerank`

Fit BM25 thật trên 572 corpus texts (scroll bounded từ Qdrant), fusion min-max 0.6/0.4 trên 30 dense candidates, trả top 10.


In [ ]:
service_hybrid = run_profile(
    "hybrid_no_rerank",
    expected_fields={"hybrid_score", "normalized_dense_score", "normalized_bm25_score"},
)


## Profile `hybrid_rerank`

Cùng hybrid pipeline, sau đó local MiniLM CrossEncoder thật rerank 10 pairs thành top 5.


In [ ]:
service_rerank = run_profile(
    "hybrid_rerank",
    expected_fields={"rerank_score", "reranker_model"},
)


## Typed failures thật qua public API

Dùng service thật đã build để kiểm tra query không hợp lệ (rỗng hoặc chỉ có khoảng trắng) bị reject bằng `InvalidQueryError`.


In [ ]:
from core.schema import InvalidQueryError

try:
    service_dense.search("   ")
except InvalidQueryError as exc:
    print("InvalidQueryError (whitespace):", exc)

try:
    service_dense.search("")
except InvalidQueryError as exc:
    print("InvalidQueryError (empty):", exc)


## Checklist xác nhận Phase 5

1. Cả ba profiles build service thật với candidate collection trong Qdrant/E5/MiniLM thật và trả documents hợp lệ.
2. `dense_only`: chỉ có `dense_score`; `hybrid_no_rerank`: có `hybrid_score` + normalized fields; `hybrid_rerank`: có `rerank_score` + `reranker_model`.
3. Cùng một câu hỏi được chạy qua cả ba profiles; không in full context.
4. Typed errors hiển thị đúng contract qua public API.
5. Không đổi config file, không benchmark, không claim winner, không tốn phí.
